# LightOnOCR-3 4B — document OCR quality upgrade on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/LightOnOCR-3-4B-document-ocr.ipynb)

[`lightonai/LightOnOCR-3-4B`](https://huggingface.co/lightonai/LightOnOCR-3-4B) is the largest LightOnOCR-3 checkpoint (about 4.5B parameters, 9.1 GB of weights, Apache 2.0) on the Qwen3.5 vision-language architecture. LightOn recommends it for most OCR tasks. An empty prompt transcribes the page to markdown. The prompt `grounding` adds a label and a bounding box, in page coordinates normalized to 0–1000, for every block: text, tables, handwriting, forms, and charts (charts come back as an HTML table of the plotted values).

This notebook is a **sibling** of [`LightOnOCR-3-0.8B-document-ocr.ipynb`](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/LightOnOCR-3-0.8B-document-ocr.ipynb), not a replacement. Same prompts, same T4 rules, a heavier model. The demos below are the page types the 4B was trained harder for: a multi-column paper, a results table, and a government form.

**When to prefer 0.8B vs 4B on a T4**

* **0.8B** — faster, fits **float16** at the card's 2048 px recipe, about 1.7 GB of weights and a 5–8 GB peak. Use it for receipts, short letters, and any page where throughput matters more than the last bit of layout quality.
* **4B** — quality upgrade. On [fr-bench-pdf2md](https://huggingface.co/blog/lightonai/lightonocr-3) it leads the 0.8B on forms (49.6 vs 47.8), handwriting (46.1 vs 33.3), long tables (83.6 vs 70.9), and multi-column pages (83.6 vs 80.6). Use it when the page is dense.

**Licence:** Apache 2.0.

**Runtime:** *Runtime > Change runtime type > T4 GPU*. The free T4 has 16 GB of VRAM, no native bfloat16, and no FlashAttention 2. The Hub file is **9.1 GB** of bfloat16 weights; float16 is the same size and leaves little room for activations on a 2048 px page. This notebook therefore defaults to **4-bit NF4** with **SDPA** attention (eager if SDPA refuses the hybrid stack) and a longest edge of **1540 px**. 1540 px is the resolution LightOn quotes for higher throughput; the card's quality recipe is still 2048 px. Set `USE_4BIT = False` and `LONGEST = 2048` only on a GPU larger than 16 GB. Figures printed after load are measured; any peak-during-generate comment is an estimate from the weight file plus a page, not a measured Colab run.

**Not in this notebook:** a LoRA fine-tune, FlashAttention 2, or loading the 0.8B checkpoint at the same time. Two models will not fit.

**Links:** [model card](https://huggingface.co/lightonai/LightOnOCR-3-4B) · [0.8B notebook](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/LightOnOCR-3-0.8B-document-ocr.ipynb) · [blog](https://huggingface.co/blog/lightonai/lightonocr-3) · [paper](https://arxiv.org/abs/2601.14251) · [demo](https://huggingface.co/spaces/lightonai/LightOnOCR-3-Demo)


In [ ]:
%pip install -q "transformers==5.20.1" "bitsandbytes>=0.45.0" "accelerate>=1.2.0" "pypdfium2==5.14.0" pillow

# The model card requires transformers>=5.5.4 (the checkpoint was saved with 5.5.4)
# and loads as Qwen3_5ForConditionalGeneration. 5.20.1 matches the 0.8B sibling.
# bitsandbytes + accelerate: 4-bit NF4. The Hub file is 9.1 GB and does not leave
# a comfortable float16 margin on a 16 GB T4.
# pypdfium2 renders the PDF pages. pillow is the image library the card lists; Colab already has it.


> **Restart the runtime now:** *Runtime > Restart session*, then continue from the next cell.
> Colab has already imported its preinstalled transformers. The upgrade is invisible until the process restarts.


In [ ]:
import torch
import transformers
from packaging.version import Version

assert Version(transformers.__version__) >= Version("5.5.4"), (
    f"transformers {transformers.__version__} is loaded; LightOnOCR-3 needs 5.5.4+. "
    "Run the install cell, then Runtime > Restart session."
)

if not torch.cuda.is_available():
    raise SystemExit("Select Runtime > Change runtime type > T4 GPU, then rerun from the top.")

# T4 (sm_75) has no FlashAttention 2. Keep PyTorch SDPA on the mem-efficient and math kernels.
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)

major, minor = torch.cuda.get_device_capability()
DTYPE = torch.bfloat16 if major >= 8 else torch.float16

# T4 defaults. On an L4 / A100 you can set USE_4BIT = False and LONGEST = 2048
# for the card's full-precision, 2048 px recipe.
USE_4BIT = major < 8
LONGEST = 1540 if USE_4BIT else 2048

print("transformers", transformers.__version__)
print("torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(), f"sm_{major}{minor}", "| dtype:", DTYPE)
print(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print(f"load: {'4-bit NF4' if USE_4BIT else 'full precision'} | longest edge: {LONGEST} px")


## 1. Load LightOnOCR-3 4B

The card loads `Qwen3_5ForConditionalGeneration` and `AutoProcessor`. Thinking stays off: the checkpoint is trained and evaluated that way, and the chat template defaults to it. The first run downloads about 9.1 GB, then bitsandbytes packs the weights to 4-bit on a T4. Compute stays in float16 (T4 has no bfloat16).


In [ ]:
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen3_5ForConditionalGeneration

MODEL_ID = "lightonai/LightOnOCR-3-4B"


def load_ocr(attn):
    kwargs = {
        "attn_implementation": attn,
        "device_map": "auto",
    }
    if USE_4BIT:
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=DTYPE,
        )
    else:
        kwargs["dtype"] = DTYPE
    return Qwen3_5ForConditionalGeneration.from_pretrained(MODEL_ID, **kwargs).eval()


# SDPA is the T4 path. Fall back to eager if this transformers build rejects SDPA
# on Qwen3.5's linear-attention layers.
try:
    model = load_ocr("sdpa")
    ATTN = "sdpa"
except Exception as exc:
    print("sdpa load failed, retrying eager:", exc)
    model = load_ocr("eager")
    ATTN = "eager"

processor = AutoProcessor.from_pretrained(MODEL_ID)
print("attention:", ATTN, "| 4-bit:", USE_4BIT, "| compute dtype:", DTYPE)
print(f"VRAM allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"VRAM reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")


## 2. Multi-column page, dense table, and a form

Samples, chosen to show why the 4B exists:

* page 1 of [Attention Is All You Need](https://arxiv.org/pdf/1706.03762) (arXiv:1706.03762) — two-column layout, title, authors, abstract
* page 8 of the same paper, which holds Table 2 — the same results table as the [0.8B notebook](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/LightOnOCR-3-0.8B-document-ocr.ipynb), so you can compare outputs without loading both models
* page 1 of [IRS Form W-9](https://www.irs.gov/pub/irs-pdf/fw9.pdf) (US government work, public domain) — a dense form with fields, checkboxes, and small instruction type

PDFs are rendered at 400 DPI and limited to `LONGEST` px on the long edge. The card's recipe is 2048 px; this T4 path uses 1540 px. The grounding prompt is the only instruction the card supports besides an empty transcription prompt.


In [ ]:
import re
import time

import requests
from IPython.display import display
from PIL import ImageDraw

PAPER_URL = "https://arxiv.org/pdf/1706.03762"
FORM_URLS = [
    "https://www.irs.gov/pub/irs-pdf/fw9.pdf",
]
UA = {"User-Agent": "donvito-notebooks/1.0 (educational Colab demo)"}
BLOCK = re.compile(r"!\[([^\]]+)\]\((\d+)\s*,\s*(\d+)\s*,\s*(\d+)\s*,\s*(\d+)\)")


def fetch_bytes(url, attempts=4):
    for attempt in range(attempts):
        response = requests.get(url, headers=UA, timeout=120)
        if response.status_code not in (429, 500, 502, 503, 504) or attempt == attempts - 1:
            response.raise_for_status()
            return response.content
        retry_after = response.headers.get("Retry-After", "")
        time.sleep(int(retry_after) if retry_after.isdigit() else 2 ** (attempt + 1))


def render_pdf_page(url, page_index, longest=LONGEST):
    import pypdfium2 as pdfium

    pdf = pdfium.PdfDocument(fetch_bytes(url))
    image = pdf[page_index].render(scale=400 / 72).to_pil().convert("RGB")
    image.thumbnail((longest, longest))
    return image


def render_pdf_page_from_urls(urls, page_index, longest=LONGEST):
    for url in urls:
        try:
            return render_pdf_page(url, page_index, longest=longest)
        except requests.RequestException as error:
            print(f"Could not fetch {url}: {error}")
    raise RuntimeError("All PDF URLs failed")


column_page = render_pdf_page(PAPER_URL, page_index=0)
table_page = render_pdf_page(PAPER_URL, page_index=7)  # printed page 8, Table 2
form_page = render_pdf_page_from_urls(FORM_URLS, page_index=0)
print("column page", column_page.size, "| table page", table_page.size, "| form", form_page.size)


def to_model(inputs):
    # 4-bit packs weights; floating inputs still run in DTYPE (float16 on a T4).
    inputs = inputs.to(model.device)
    for key, value in list(inputs.items()):
        if hasattr(value, "is_floating_point") and value.is_floating_point():
            inputs[key] = value.to(dtype=DTYPE)
    return inputs


def transcribe(image, prompt, max_new_tokens=1024):
    if isinstance(image, str):
        content = [{"type": "image", "url": image}]
    else:
        content = [{"type": "image", "image": image}]
    if prompt:
        content.append({"type": "text", "text": prompt})
    conversation = [{"role": "user", "content": content}]
    inputs = processor.apply_chat_template(
        conversation,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        enable_thinking=False,
    )
    inputs = to_model(inputs)
    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.2,
        )
    generated = output_ids[0, inputs["input_ids"].shape[1] :]
    text = processor.decode(generated, skip_special_tokens=True)
    if torch.cuda.is_available():
        print(f"VRAM allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
        print(f"VRAM reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")
    return text


def draw_grounding(image, text):
    canvas = image.copy()
    draw = ImageDraw.Draw(canvas)
    width, height = canvas.size
    hits = list(BLOCK.finditer(text))
    for match in hits:
        label, x1, y1, x2, y2 = match.groups()
        box = (
            int(x1) / 1000 * width,
            int(y1) / 1000 * height,
            int(x2) / 1000 * width,
            int(y2) / 1000 * height,
        )
        draw.rectangle(box, outline=(220, 40, 40), width=3)
        draw.text((box[0] + 2, box[1] + 2), label, fill=(220, 40, 40))
    return canvas, len(hits)


### Multi-column — page 1 of Attention Is All You Need

Two columns, a title block, and an abstract. The 0.8B sibling starts on a receipt; this page is closer to the layouts where the 4B pulls ahead. Grounding should label title, authors, and the two text columns.


In [ ]:
display(column_page)
column_text = transcribe(column_page, "grounding", max_new_tokens=1200)
print(column_text)
boxed, n_boxes = draw_grounding(column_page, column_text)
print(f"{n_boxes} grounding boxes")
display(boxed)


### Dense table — page 8 of Attention Is All You Need

Same Table 2 page as the 0.8B notebook. Open that notebook in another tab if you want a side-by-side read of the markdown; do not load 0.8B in this runtime. Grounding should mark the table block and keep the BLEU / perplexity cells aligned.


In [ ]:
display(table_page)
table_text = transcribe(table_page, "grounding", max_new_tokens=1200)
print(table_text)
boxed, n_boxes = draw_grounding(table_page, table_text)
print(f"{n_boxes} grounding boxes")
display(boxed)


### Form — IRS W-9 (Rev. March 2024), page 1

A blank government form: numbered fields, checkboxes, and small instruction type. Forms are one of the categories where the 4B leads the 0.8B on fr-bench-pdf2md. Source: [IRS](https://www.irs.gov/pub/irs-pdf/fw9.pdf), US government work, public domain.


In [ ]:
print("Source: IRS Form W-9, US government work, public domain.")
display(form_page)
form_text = transcribe(form_page, "grounding", max_new_tokens=1400)
print(form_text)
boxed, n_boxes = draw_grounding(form_page, form_text)
print(f"{n_boxes} grounding boxes")
display(boxed)


## 3. 0.8B vs 4B — quality and speed on a T4

Do **not** load [`lightonai/LightOnOCR-3-0.8B`](https://huggingface.co/lightonai/LightOnOCR-3-0.8B) in this session. The 0.8B notebook is the place to run that checkpoint.

| | 0.8B sibling | 4B (this notebook) |
| --- | --- | --- |
| Hub weights | ~1.7 GB | ~9.1 GB |
| Parameters | ~0.85B | ~4.5B |
| T4 default | float16, 2048 px, SDPA | 4-bit NF4, 1540 px, SDPA |
| Comfortable peak (estimate) | 5–8 GB | 4-bit weights plus a 1540 px page; float16 + 2048 px is tight on 16 GB |
| Role | Fast / efficient | Quality upgrade, LightOn's default for most tasks |
| Typical win | Receipts, short pages, throughput | Dense tables, forms, multi-column, handwriting |

Expect the 4B to be slower per page on a T4: the model is larger, and 4-bit dequantizes during compute. The quality gap is small on a clean receipt and larger on cramped fields, long tables, and handwriting (fr-bench handwritten 46.1 vs 33.3). Other instructions besides `""` and `grounding` are out of distribution.


## Caveats

* **VRAM.** Float16 weights are ~9.1 GB. The T4 path is 4-bit NF4 and a 1540 px longest edge. Raise `LONGEST` to 2048 only if reserved memory after load still leaves several GB of headroom.
* **dtype.** T4 has no native bfloat16. Compute dtype is float16. Do not set `torch.bfloat16` on this GPU.
* **Attention.** SDPA, or eager if SDPA rejects Qwen3.5's linear-attention layers. Not FlashAttention 2.
* **Licence.** Apache 2.0.
* **Prompts.** Empty string or `grounding` only.
